# 1、ChatMessageHistory的使用

场景1：记忆存储


In [ ]:
from dotenv import load_dotenv
# from langchain_community.chat_message_histories import ChatMessageHistory
# ✅ 推荐写法 (最新标准)
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1、ChatMessageHistory的实例化
history = InMemoryChatMessageHistory()
#
# # 2、添加相关的消息进行存储
history.add_user_message("你好")
history.add_ai_message("很高兴认识你")

# # 3、打印存储的消息
print(history.messages)


场景2：对接LLM



In [2]:
# 1、获取大模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# from langchain.memory import ChatMessageHistory
# ✅ 推荐写法 (最新标准)
from langchain_core.chat_history import InMemoryChatMessageHistory
# 1、ChatMessageHistory的实例化

history = InMemoryChatMessageHistory()

# 2、添加相关的消息进行存储
history.add_user_message("你好")
history.add_ai_message("很高兴认识你")
history.add_user_message("帮我计算1 + 2 * 3 = ？")

response = llm.invoke(history.messages)
print(response.content)

# 2、ConversationBufferMemory的使用

举例1：以字符串的方式返回存储的信息

In [ ]:
from langchain.memory import ConversationBufferMemory

# 1、ConversationBufferMemory的实例化
memory = ConversationBufferMemory()

# 2、存储相关的消息
# inputs对应的就是用户消息，outputs对应的就是ai消息
memory.save_context(inputs={"human": "你好，我叫小明"}, outputs={"ai": "很高兴认识你"})
memory.save_context(inputs={"input": "帮我回答一下1+2*3=?"}, outputs={"output": "7"})

# 3、获取存储的信息
print(memory.load_memory_variables({}))

#说明：返回的字典结构的key叫history.

举例2：以消息列表的方式返回存储的信息

In [ ]:
from langchain.memory.buffer import ConversationBufferMemory
# 1、ConversationBufferMemory的实例化
memory = ConversationBufferMemory(return_messages=True)

# 2、存储相关的消息
# inputs对应的就是用户消息，outputs对应的就是ai消息
memory.save_context(inputs={"human": "你好，我叫小明"}, outputs={"ai": "很高兴认识你"})
memory.save_context(inputs={"input": "帮我回答一下1+2*3=?"}, outputs={"output": "7"})

# 3、获取存储的信息
#返回消息列表的方式1：
print(memory.load_memory_variables({}))

print("\n")

#返回消息列表的方式2：
print(memory.chat_memory.messages)

#说明：返回的字典结构的key叫history.

举例3：结合大模型、提示词模板的使用（PromptTemplate）

In [ ]:
from langchain.chains.llm import LLMChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人类对话。

当前对话历史: {history}

人类问题: {question}

回复:
"""
)

# 3、提供memory实例
memory = ConversationBufferMemory()

# 4、提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

response = chain.invoke({"question": "你好，我的名字叫小明"})
print(response)

In [ ]:
response = chain.invoke({"question": "我叫什么名字呢？"})
print(response)

举例4：基于举例3，显式的设置meory的key的值


In [ ]:
from langchain.chains.llm import LLMChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate

# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人类对话。

当前对话历史: {chat_history}

人类问题: {question}

回复:
"""
)

# 3、提供memory实例
memory = ConversationBufferMemory(memory_key="chat_history")

# 4、提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

response = chain.invoke({"question": "你好，我的名字叫小明"})
print(response)

In [ ]:
response = chain.invoke({"question": "我叫什么名字呢？"})
print(response)

举例5：结合大模型、提示词模板的使用（ChatPromptTemplate）

In [ ]:
# 1.导入相关包
from langchain_core.messages import SystemMessage
from langchain.chains.llm import LLMChain
from langchain.memory.buffer import ConversationBufferMemory
from langchain_core.prompts import MessagesPlaceholder,ChatPromptTemplate,HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI


# 2.创建LLM
llm = ChatOpenAI(model_name='gpt-4o-mini')

# 3.创建Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system","你是一个与人类对话的机器人。"),
    MessagesPlaceholder(variable_name='history'),
    ("human","问题：{question}")
])

# 4.创建Memory
memory = ConversationBufferMemory(return_messages=True)
# 5.创建LLMChain
llm_chain = LLMChain(prompt=prompt,llm=llm, memory=memory)

# 6.调用LLMChain
res1 = llm_chain.invoke({"question": "中国首都在哪里？"})
print(res1,end="\n\n")



In [ ]:
res2 = llm_chain.invoke({"question": "我刚刚问了什么"})
print(res2)


# 3、ConversationChain的使用

举例1：以PromptTemplate为例

In [ ]:
from langchain.chains.conversation.base import ConversationChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate

# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
     你可以与人类对话。
     当前对话历史: {history}
     人类问题: {input}
     回复:
    """
)

# # 3、提供memory实例
# memory = ConversationBufferMemory()
#
# # 4、提供Chain
# chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

# 3、创建ConversationChain的实例
chain = ConversationChain(llm = llm, prompt=prompt_template)

response = chain.invoke({"input": "你好，我的名字叫小明"})
print(response)

In [ ]:
response = chain.invoke({"input": "我的名字叫什么？"})
print(response)

# 基于最新的langchain的版本 重写的代码


In [3]:
# 基于最新的langchain的版本 重写的代码

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

# 1、创建大模型实例
dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

llm = ChatOpenAI(model="gpt-4o-mini")

# 2.创建Prompt 注意 新版本推荐使用ChatPromptTemplate
# MessagesPlaceHolder(variable_name="history") 是关键 它告诉我们LangChain 在哪里插入历史记录

prompt = ChatPromptTemplate.from_messages([
    ("system","你可以与人类对话。"),
    MessagesPlaceholder(variable_name='history'), # 这里回自动填充历史消息
    ("human","{input}")
])

# 3. 定义 chain 使用LCEL 管道语法
chain = prompt | llm
# 4. 管理记忆(Memory)
# 我们需要一个函数来根据 session_id 获取对应的历史记录
store = {}

def get_session_history(session_id:str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 5. 创建带历史记录的Runnable
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input", # 对应 prompt 中的 human input
    history_messages_key="history", # 对应 prompt 中的 MessagesPlaceholder
)

# 6. 调用 需要传入 session_id 来区分不同用户的对话
# 第一轮对话
response1 = with_message_history.invoke(
    {"input": "你好，我的名字叫小明"},
    config={"configurable": {"session_id": "user_123"}}
)
print(f"AI回复1: {response1.content}")

# 第二轮对话 (验证记忆)
response2 = with_message_history.invoke(
    {"input": "我刚才叫什么名字？"},
    config={"configurable": {"session_id": "user_123"}} # 使用相同的 session_id
)
print(f"AI回复2: {response2.content}")

AI回复1: 你好，小明！很高兴认识你。你今天过得怎么样？
AI回复2: 你刚才说你的名字叫小明。有什么我可以帮助你的吗？


## 主要变化点解析：

不再使用 ConversationChain：
以前的 ConversationChain 把 LLM、Prompt 和 Memory 绑死在了一起。现在的做法是将它们解耦。
Prompt 的变化：
使用了 ChatPromptTemplate。以前你需要在字符串里写 {history}，现在使用 MessagesPlaceholder(variable_name="history")。这对于 Chat 模型（如 gpt-4o）来说更准确，因为它会把历史记录作为 SystemMessage, AIMessage, HumanMessage 的列表插入，而不是仅仅拼接成一坨字符串。
chain = prompt | llm：
这是 LCEL 语法。所有的处理流程都变成了“管道”操作。
RunnableWithMessageHistory：
这是核心替代品。
它是一个“包装器”，包裹了你的 chain。
你需要提供一个 get_session_history 函数。这意味着现在的记忆管理天然支持多用户/多会话（通过 session_id 区分），而以前的 ConversationBufferMemory 默认只能存一个会话。
调用方式 invoke：
调用时多了一个 config 参数，用来指定 session_id。

pip install langchain-community langchain-core langchain-openai


举例2：使用默认提供的提示词模板



In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain.memory.buffer import ConversationBufferMemory
from langchain.chains.llm import LLMChain
from langchain.chains.conversation.base import ConversationChain

# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人类对话。

当前对话历史: {history}

人类问题: {input}

回复:
"""
)

# 3、提供memory实例
memory = ConversationBufferMemory()
# 4、提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

# 3、创建ConversationChain的实例（内部提供了默认的提示词模板。而此模板中的变量是{input}、{history}
chain = ConversationChain(llm = llm)

response = chain.invoke({"input": "你好，我的名字叫小明"})
print(response)

/var/folders/gd/xcfqj9752391g13gs68sxcfr0000gn/T/ipykernel_13246/2578650274.py:29: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html instead.
  chain = ConversationChain(llm = llm)


{'input': '你好，我的名字叫小明', 'history': '', 'response': '你好，小明！很高兴认识你！我叫AI助手。你今天过得怎么样？有什么特别的事情想分享吗？'}


In [ ]:
response = chain.invoke({"input": "我的名字叫什么？"})
print(response)

## 举例2：使用默认提供的提示词模板 最新版本的langchain的实现


在 LangChain 0.2/0.3+ 的最新版本中，ConversationChain 已经被废弃。现在官方推荐使用 LCEL (LangChain Expression Language) 结合 RunnableWithMessageHistory 来实现带有记忆的对话。

为了实现“默认提示词模板”的效果（即一个标准的、友好的对话 AI），我们需要构建一个包含 System Message（系统设定）、History Placeholder（历史记录占位符）和 Human Message（用户输入）的 ChatPromptTemplate。

以下是使用最新标准重写的代码：
核心变更点:

1. 移除 ConversationChain, LLMChain, ConversationBufferMemory。
2. 使用 RunnableWithMessageHistory 来管理记忆。
3. 使用 InMemoryChatMessageHistory 来存储消息。
4. 使用 MessagesPlaceholder 自动填充历史记录。

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory

# 1. 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2. 创建提示词模版 模拟 ConversationChain 的默认风格
# 以前 ConversationChain 的默认模板大约是 "The following is a friendly conversation..."
# 在 Chat 模型中，我们将其转换为 System Message

prompt = ChatPromptTemplate.from_messages([
    # 系统消息 定义AI的基本行为 相当于默认模版的设定部分
    ("system","The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know."),
    # 历史记录占位符：LangChain 会自动把取到的历史消息列表填充在这里
    MessagesPlaceholder(variable_name='history'),
    # 人类输入
    ("human","{input}")
])

# 3. 定义基础 Chain
chain = prompt | llm

# 4. 定义记忆存储管理
# 用于在内存中存储不同 session_id 的对话历史
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 5. 创建带历史记忆的 Runnable 实例
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",   # 对应 prompt 中的 {input}
    history_messages_key="history", # 对应 prompt 中的 MessagesPlaceholder
)

# 6. 调用
# 必须配置 session_id，以便区分不同用户的对话
config = {"configurable": {"session_id": "abc1234"}}

# 第一轮对话
response1 = with_message_history.invoke(
    {"input": "你好，我的名字叫小明"},
    config=config
)
print(f"AI回复1: {response1.content}")

# 第二轮对话 (测试是否记住了名字)
response2 = with_message_history.invoke(
    {"input": "我刚才叫什么名字？"},
    config=config
)
print(f"AI回复2: {response2.content}")

AI回复1: 你好，小明！很高兴认识你！你今天过得怎么样？有什么想聊的或者想了解的事情吗？
AI回复2: 你刚才说你的名字叫小明！这是个很不错的名字。你喜欢这个名字吗？


## 代码解析
1. ChatPromptTemplate:
- 以前的默认模板是一个长字符串。现在针对 Chat 模型，我们将“人设”放在 ("system", ...) 中。
- MessagesPlaceholder(variable_name="history") 是最关键的部分，它替代了以前字符串里的 {history}，它能完美处理 UserMessage 和 AIMessage 的对象列表。
2. InMemoryChatMessageHistory:
这是新版 langchain-core 提供的轻量级内存历史存储，替代了旧的 ConversationBufferMemory 在这里的用法。
3. session_id:
新版设计天然支持多用户。你需要传入 session_id 来告诉程序当前是哪个用户在说话。如果你只是单人测试，随便写一个 ID 即可（如代码中的 "abc1234"）。

# 4、ConversationBufferWindowMemory的使用

ConversationBufferWindowMemory 是langchain中一种用于管理对话历史的组件，他的核心特点是 滑动窗口 机制： 它只是保留最近 k 轮对话，旧的对话会被自动丢弃。

这主要用于控制 Token 消耗，防止随着对话变长，历史记录超出大模型的上下文限制。

举例1：


In [12]:
# 1.导入相关包
from langchain.memory import ConversationBufferWindowMemory

# 2.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=1)
# 3.保存消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手"})
memory.save_context({"input": "你的生日是哪天？"}, {"output": "我不清楚"})
# 4.读取内存中消息（返回消息内容的纯文本）
print(memory.load_memory_variables({}))

{'history': 'Human: 你的生日是哪天？\nAI: 我不清楚'}


详细过程分析：

1. k=1 (设置窗口):
你实例化时设置了 k=1。这意味着内存中只保留最近的 1 轮对话（即 1 条 Human 消息 + 1 条 AI 消息）。
2. 保存过程:
第 1 次 save: 存入 "你好" / "怎么了"。 (内存中: 1 轮)
第 2 次 save: 存入 "你是谁" / "我是AI助手"。
由于 k=1，第 1 轮被挤出，内存中现在是: "你是谁" / "我是AI助手"。
第 3 次 save: 存入 "你的生日是哪天？" / "我不清楚"。
由于 k=1，第 2 轮被挤出，内存中现在是: "你的生日是哪天？" / "我不清楚"。
3. 读取过程:
调用 load_memory_variables 时，只返回当前内存中剩下的那 1 轮对话。

## 2. 进阶：在 LangChain 0.2/0.3+ 中的实现 (LCEL)

虽然 ConversationBufferWindowMemory 在旧版中很常用，但在新版 LangChain (LCEL) 架构中，官方推荐将 “存储” 和 “裁剪逻辑” 分开。

我们使用 RunnableWithMessageHistory 来存储所有历史，但在传给大模型之前，使用 trim_messages 来实现“窗口”效果。这样做的好处是数据库里可以存所有的聊天记录（用于审计），但发给 LLM 的只有最近几条（省钱）。

## 现代写法 (实现 k=1 的滑动窗口)：

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import trim_messages

# 1. 模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 2. 定义裁剪器 (Trimmer) - 这就是新的 "WindowMemory"
# 这里我们设定 strategy="last"，并限制 max_tokens 来模拟窗口
# 或者直接按消息数量裁剪 (注意：token计算更精准，但这里为了演示 k 的概念，我们保留最后 2 条消息，即一问一答)
trimmer = trim_messages(
    strategy="last",
    token_counter=llm,             # 使用模型的 token 计算器
    max_tokens=200,                # 限制最近的 200 个 token (或者根据消息数裁剪)
    start_on="human",              # 确保截取后的第一条是人类说的话
    include_system=True,           # 保留系统提示词
    allow_partial=False,
)

# 3. Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system","你是一个助手"),
    MessagesPlaceholder(variable_name='history'),
    ("human","{input}")
])

# 4. chain 加入 trimmer
# 逻辑 ： 获取历史——> 裁剪历史 -> 放入Prompt -> 传给LLM

chain = RunnableWithMessageHistory(
    prompt | llm ,
    get_session_history=lambda session_id: InMemoryChatMessageHistory(), # 每次运行实例用新内存 实际用全局store
    input_messages_key="input",
    history_messages_key="history",
)

# 注意 ： 上面的写法会讲所有历史注入
# 在 LCEL 中实现严格的 Window Memory 通常是在 Prompt内部或Chain 构造时加入 trim 逻辑
# 下面是一个更直观的、手动实现 "只取最近 k 轮" 的完整 LCEL 例子：

#####################################################################
# 推荐的完整 LCEL "Window Memory" 实现
#####################################################################

# 存储
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 核心逻辑 ： 定义一个 Chain 它接受输入 自动裁剪历史 然后调用模型
prompt = ChatPromptTemplate.format_messages([
   ("system", "你是一个助手"),
    MessagesPlaceholder(variable_name="history"), # 这里填充裁剪后的历史
    ("human", "{input}"),
])

# 定义 Trim 逻辑
# 注意：last_max_tokens 是新版控制窗口大小的标准方式，比 k 更科学
trimmer = trim_messages(
    strategy="last",
    token_counter=llm,
    max_tokens=100, # 限制上下文大小，相当于变相的 k
    start_on="human",
)
chain = prompt | llm

# 使用 RunnableWithMessageHistory 包装，但在注入 prompt 之前，我们需要拦截 history 进行裁剪
# 由于 RunnableWithMessageHistory 封装较深，更灵活的方式是：
# 1. 有一个 PassThrough 负责取历史
# 2. trimmer 裁剪
# 3. 传给 prompt

# 但为了最简便，如果只是想复现 k=1：
final_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 调用时，trim_messages 通常作为 chain 的一部分被手动调用，或者使用 memory 自动管理。
# 简单来说，ConversationBufferWindowMemory 依然可以用，但在 LCEL 中，
# 你可以直接在 RunnableWithMessageHistory 内部处理，
# 或者不做处理，依靠 LLM 上下文限制自动截断（不推荐）。

### 总结
ConversationBufferWindowMemory(k=N):
作用: 滑动窗口记忆。
逻辑: 只存最新的 N 组对话，旧的删掉。
场景: 节省 Token，让 AI 即使聊了很久也不会报错，但会忘记很久之前说的话。

举例2：返回消息构成的上下文记忆




In [ ]:
# 1.导入相关包


# 2.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=2, return_messages=True)
# 3.保存消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手小智"})
memory.save_context({"input": "初次对话，你能介绍一下你自己吗？"}, {"output": "当然可以了。我是一个无所不能的小智。"})
# 4.读取内存中消息（返回消息内容的纯文本）
print(memory.load_memory_variables({}))

举例3：结合llm、chain的使用


In [15]:

# 1.导入相关包
from langchain_core.prompts.prompt import PromptTemplate


# 2.定义模版
template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的具体细节。如果AI不知道问题的答案，它会表示不知道。

当前对话：
{history}
Human: {question}
AI:"""

# 3.定义提示词模版
prompt_template = PromptTemplate.from_template(template)

# 4.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 5.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=1)

# 6.定义LLMChain
conversation_with_summary = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    #verbose=True,
)

# 7.执行链（第一次提问）
respon1 = conversation_with_summary.invoke({"question":"你好，我是孙小空"})
print(respon1)
# 8.执行链（第二次提问）
respon2 =conversation_with_summary.invoke({"question":"我还有两个师弟，一个是猪小戒，一个是沙小僧"})
print(respon2)
# 9.执行链（第三次提问）
respon3 =conversation_with_summary.invoke({"question":"我今年高考，竟然考上了1本"})
print(respon3)
# 10.执行链（第四次提问）
respon4 =conversation_with_summary.invoke({"question":"我叫什么名字？"})
print(respon4)

{'question': '你好，我是孙小空', 'history': '', 'text': '你好，孙小空！很高兴认识你！你今天过得怎么样？有什么我可以帮助你的吗？'}
{'question': '我还有两个师弟，一个是猪小戒，一个是沙小僧', 'history': 'Human: 你好，我是孙小空\nAI: 你好，孙小空！很高兴认识你！你今天过得怎么样？有什么我可以帮助你的吗？', 'text': '很高兴认识你的师弟们，孙小空！猪小戒和沙小僧的名字听起来很有趣，像是来自某个故事或者传说。你们三个人有什么共同的爱好或者活动吗？'}
{'question': '我今年高考，竟然考上了1本', 'history': 'Human: 我还有两个师弟，一个是猪小戒，一个是沙小僧\nAI: 很高兴认识你的师弟们，孙小空！猪小戒和沙小僧的名字听起来很有趣，像是来自某个故事或者传说。你们三个人有什么共同的爱好或者活动吗？', 'text': '太棒了！恭喜你考上了一本！这是一个很大的成就，值得庆祝。你打算选择什么专业呢？或者你有考虑过未来的职业方向吗？'}
{'question': '我叫什么名字？', 'history': 'Human: 我今年高考，竟然考上了1本\nAI: 太棒了！恭喜你考上了一本！这是一个很大的成就，值得庆祝。你打算选择什么专业呢？或者你有考虑过未来的职业方向吗？', 'text': '抱歉，我不知道你的名字。如果你愿意，可以告诉我你的名字，或者我们可以继续聊其他话题！你对大学生活有什么期待吗？'}


举例4：修改举例3中的参数k


In [14]:
# 1.导入相关包
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts.prompt import PromptTemplate
from langchain.chains.llm import LLMChain


# 2.定义模版
template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的具体细节。如果AI不知道问题的答案，它会表示不知道。

当前对话：
{history}
Human: {question}
AI:"""

# 3.定义提示词模版
prompt_template = PromptTemplate.from_template(template)

# 4.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 5.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=3)

# 6.定义LLMChain
conversation_with_summary = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    #verbose=True,
)

# 7.执行链（第一次提问）
respon1 = conversation_with_summary.invoke({"question":"你好，我是孙小空"})
# print(respon1)
# 8.执行链（第二次提问）
respon2 =conversation_with_summary.invoke({"question":"我还有两个师弟，一个是猪小戒，一个是沙小僧"})
# print(respon2)
# 9.执行链（第三次提问）
respon3 =conversation_with_summary.invoke({"question":"我今年高考，竟然考上了1本"})
# print(respon3)
# 10.执行链（第四次提问）
respon4 =conversation_with_summary.invoke({"question":"我叫什么名字？"})
print(respon4)

{'question': '我叫什么名字？', 'history': 'Human: 你好，我是孙小空\nAI: 你好，孙小空！很高兴见到你。你今天过得怎么样？有什么想聊的话题吗？\nHuman: 我还有两个师弟，一个是猪小戒，一个是沙小僧\nAI: 哇，听起来你们的名字都很有趣！猪小戒和沙小僧分别代表了《西游记》中的猪八戒和沙和尚吗？你们是一起学习还是有其他的活动？\nHuman: 我今年高考，竟然考上了1本\nAI: 太棒了，恭喜你考上了1本！这是一个很大的成就，你一定付出了很多努力。你打算选择什么专业呢？或者你对未来有什么计划吗？', 'text': '你叫孙小空！如果你有其他问题或者想聊的话题，请随时告诉我！'}
